## Memory 챌린지

In [1]:
from langchain.chat_models import ChatOpenAI
from langchain.memory import ConversationSummaryBufferMemory
from langchain.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate, MessagesPlaceholder
from langchain.schema.runnable import RunnablePassthrough

llm = ChatOpenAI(
    temperature=0.1
)

examples = [
    {
        "movie": "Top Gun",
        "answer": "🛩️👨‍✈️🔥",
    },
    {
        "movie": "The Godfather",
        "answer": "👨‍👨‍👦🔫🍝",
    },
    {
        "movie": "Titanic",
        "answer": "🚢🧊💔",
    },
    {
        "movie": "Harry Potter",
        "answer": "🧙‍♂️⚡🏰",
    },
    {
        "movie": "Jurassic Park",
        "answer": "🦖🏝️🚙",
    },
]

example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{movie}"),
    ("ai", "{answer}")
])

few_shot_prompt = FewShotChatMessagePromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
)

memory = ConversationSummaryBufferMemory(
    llm=llm,
    max_token_limit=120,
    return_messages=True,
)

final_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are a helpful AI talking to a human.

If the user's input is the title of a movie or refers to a movie,
follow the format shown in the examples.

For any other question, answer normally.

Use the chat history when necessary to answer questions
about the previous conversation.
        """
    ),
    few_shot_prompt,
    MessagesPlaceholder(variable_name="history"),
    ("human", "{question}"),
])

def load_memory(_):
    return memory.load_memory_variables({})["history"]

chain = RunnablePassthrough.assign(history=load_memory) | final_prompt | llm

def invoke_chain(question):
    result = chain.invoke({
        "question": question
    })
  
    memory.save_context({"input": question}, {"output": result.content},)

    print(result.content)

In [2]:
invoke_chain("Spider-Man")

🕷️🕸️🦸‍♂️


In [3]:
invoke_chain("Interstellar")

🚀🌌⏳


In [4]:
invoke_chain("The Dark Knight")

🦇🃏🦸‍♂️


In [6]:
invoke_chain("The Odyssey")

🏛️🌊🧔🏽


In [8]:
invoke_chain("What was the first movie I asked you about?")

You first asked about "Spider-Man."
